In [1]:
print("Testing notebook!!")

Testing notebook!!


In [2]:
"""
inspect_data.py
----------------
Stage 1 of the Business Entity Resolution pipeline: EDA / data profiling.

Reads Source 1 / Source 2 / Source 3 (and, for the train split, the ground
truth file), and prints schema, missingness, duplicates, text-length stats,
country distribution, and ground-truth match-count distribution.

USAGE (from anywhere, paths are absolute):

    # Quick smoke test on the train split (default: first 50,000 rows/file)
    python inspect_data.py --split train

    # Same, but on the test split
    python inspect_data.py --split test

    # Full file, once the smoke test looks good (no sampling)
    python inspect_data.py --split train --sample 0
    python inspect_data.py --split test --sample 0

If your folder layout differs from what's hardcoded below, override with:
    python inspect_data.py --root "C:\\path\\to\\dataset" --split train
"""

import argparse
import os
import re
import unicodedata

import pandas as pd


# ----------------------------------------------------------------------
# Small helpers
# ----------------------------------------------------------------------

def count_lines_fast(path):
    """Count lines in a file without loading it into memory (for reporting
    'this sample is N of M total rows'). Returns None on failure."""
    try:
        count = 0
        with open(path, "rb") as f:
            for _ in f:
                count += 1
        return count - 1  # minus header row
    except Exception:
        return None


def load_tsv(path, nrows=None):
    """Load a TSV as all-string columns (dtype=str). We deliberately avoid
    letting pandas guess int/float/date types here: guessed types can
    silently coerce things like postal codes ('00501') into numbers and
    lose leading zeros, which would corrupt IDs and codes. We normalize
    types ourselves later, on purpose, once we know what each column means.
    """
    return pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        nrows=nrows,
        keep_default_na=True,
        na_values=["", "NA", "NaN", "null", "NULL"],
    )


def normalize_name_quick(s):
    """A very light preview-only normalizer (lowercase, strip accents,
    strip punctuation, collapse whitespace). This is NOT the final
    normalization pipeline -- just enough to measure 'how much exact
    overlap exists after trivial cleanup', which tells us how hard
    blocking will be. The real normalization stage comes later."""
    if pd.isna(s):
        return None
    s = str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


# ----------------------------------------------------------------------
# Profiling functions
# ----------------------------------------------------------------------

def basic_profile(df, label):
    print(f"\n{'=' * 70}\n{label}\n{'=' * 70}")
    print(f"Rows loaded: {len(df):,}")
    print(f"Columns ({len(df.columns)}): {list(df.columns)}")

    print("\nMissing values per column:")
    miss = df.isna().sum()
    for col in df.columns:
        pct = miss[col] / len(df) * 100 if len(df) else 0
        print(f"  {col:30s} {miss[col]:>10,}  ({pct:5.2f}%)")

    id_cols = [c for c in df.columns if "entity_id" in c.lower() or c.lower() == "id"]
    if id_cols:
        idc = id_cols[0]
        dup = df[idc].duplicated().sum()
        print(f"\nDuplicate values in id column '{idc}': {dup:,}")
    else:
        print("\n(no obvious id column detected by name)")

    full_dup = df.duplicated().sum()
    print(f"Fully duplicate rows (all columns identical): {full_dup:,}")


def text_length_stats(df, col):
    if col not in df.columns:
        return
    lengths = df[col].dropna().astype(str).str.len()
    if len(lengths) == 0:
        print(f"\n'{col}': all values missing, skipping length stats")
        return
    print(f"\n'{col}' length stats (characters):")
    print(
        f"  min={lengths.min()}  p25={lengths.quantile(.25):.0f}  "
        f"median={lengths.median():.0f}  p75={lengths.quantile(.75):.0f}  "
        f"max={lengths.max()}  mean={lengths.mean():.1f}"
    )


def country_distribution(df):
    matches = [c for c in df.columns if "country" in c.lower()]
    if not matches:
        print("\n(no country-like column found)")
        return
    col = matches[0]
    vc = df[col].value_counts(dropna=False)
    print(f"\nCountry distribution (column '{col}'):")
    print(vc.head(20).to_string())


def exact_overlap_check(s1_df, other_df, other_label):
    """Cheap proxy for 'how much of this problem is easy'. Normalizes the
    first name-like and address-like column of each df and reports how
    many S1 rows have an exact normalized match in the other source."""
    s1_name_cols = [c for c in s1_df.columns if "name" in c.lower()]
    other_name_cols = [c for c in other_df.columns if "name" in c.lower()]
    if not s1_name_cols or not other_name_cols:
        print(f"\n(skipping S1 vs {other_label} overlap check -- no name column found)")
        return

    s1_norm = s1_df[s1_name_cols[0]].apply(normalize_name_quick)
    other_norm = set(other_df[other_name_cols[0]].apply(normalize_name_quick).dropna())

    hit = s1_norm.dropna().isin(other_norm).sum()
    total = s1_norm.dropna().shape[0]
    pct = hit / total * 100 if total else 0
    print(
        f"\nS1 rows with an EXACT normalized-name match somewhere in {other_label}: "
        f"{hit:,} / {total:,} ({pct:.2f}%)"
    )


def ground_truth_profile(df):
    print(f"\n{'=' * 70}\nGROUND TRUTH: MATCH-COUNT ANALYSIS\n{'=' * 70}")
    print(f"Columns: {list(df.columns)}")

    s1_cols = [c for c in df.columns if "source1" in c.lower() or c.lower() in ("s1_id", "s1")]
    match_cols = [
        c for c in df.columns
        if "match" in c.lower() and c not in s1_cols
    ]

    print(f"Detected S1-id column: {s1_cols}")
    print(f"Detected matched-ids column: {match_cols}")

    if not s1_cols:
        print(
            "\nCould not auto-detect the S1 id column by name. "
            "Please paste the actual column names and a couple of sample "
            "rows (e.g. df.head().to_string()) and I'll adjust the script."
        )
        return

    s1c = s1_cols[0]

    if match_cols:
        # WIDE format: one row per S1 entity, comma-separated match list
        mc = match_cols[0]

        def count_ids(x):
            if pd.isna(x) or str(x).strip() == "":
                return 0
            return len(str(x).split(","))

        counts = df[mc].apply(count_ids)
        print("\nFormat detected: WIDE (one row per S1 entity)")
        print("\nMatch-count distribution per S1 entity:")
        print(counts.value_counts().sort_index().head(20).to_string())
        print(f"\n% singleton (0 matches):   {(counts == 0).mean() * 100:.2f}%")
        print(f"% single match (1):        {(counts == 1).mean() * 100:.2f}%")
        print(f"% multi-match (2+):        {(counts >= 2).mean() * 100:.2f}%")
        print(f"Max matches for one S1 entity: {counts.max()}")
    else:
        # Possibly LONG format: one row per (S1, matched_id) pair
        print("\nNo single 'matched ids' column found.")
        print("Checking if this is LONG format (one row per S1<->match pair)...")
        per_s1_counts = df[s1c].value_counts()
        print("\nRows per S1 id -- distribution:")
        print(per_s1_counts.value_counts().sort_index().head(20).to_string())
        n_unique_s1 = df[s1c].nunique()
        print(f"\nUnique S1 ids in ground truth: {n_unique_s1:,}")
        print(f"Total ground-truth rows: {len(df):,}")
        print(
            "\nIf this looks like one row per pair, please paste df.head(10) "
            "so we can confirm the exact column meaning before building labels."
        )


# ----------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------

def main():
    ap = argparse.ArgumentParser(description="EDA / profiling for entity resolution data")
    ap.add_argument(
        "--root",
        default=r"C:\Users\chinm\Downloads\dataset\student_resource\dataset",
        help="Path to the 'dataset' folder containing train/ and test/",
    )
    ap.add_argument("--split", choices=["train", "test"], default="train")
    ap.add_argument(
        "--sample",
        type=int,
        default=50000,
        help="Rows to read per file for a quick smoke test. Use 0 to read the FULL file.",
    )
    args = ap.parse_args()

    split_dir = os.path.join(args.root, args.split)
    nrows = None if args.sample == 0 else args.sample

    files = {
        "source1": os.path.join(split_dir, f"{args.split}_source1.tsv"),
        "source2": os.path.join(split_dir, f"{args.split}_source2.tsv"),
        "source3": os.path.join(split_dir, f"{args.split}_source3.tsv"),
    }
    if args.split == "train":
        files["ground_truth"] = os.path.join(split_dir, "train_ground_truth.tsv")

    print(f"Split: {args.split}")
    print(f"Sample size per file: {'FULL FILE' if nrows is None else f'{nrows:,} rows'}")

    dfs = {}
    for key, path in files.items():
        print(f"\nLoading {key} <- {path}")
        if not os.path.exists(path):
            print(f"  !! FILE NOT FOUND: {path}")
            continue
        total_lines = count_lines_fast(path)
        df = load_tsv(path, nrows=nrows)
        dfs[key] = df
        extra = f" (file has {total_lines:,} total data rows)" if total_lines is not None else ""
        print(f"  Loaded {len(df):,} rows{extra}")

    for key in ["source1", "source2", "source3"]:
        if key not in dfs:
            continue
        df = dfs[key]
        basic_profile(df, key.upper())
        for c in [c for c in df.columns if "name" in c.lower()]:
            text_length_stats(df, c)
        for c in [c for c in df.columns if "address" in c.lower()]:
            text_length_stats(df, c)
        country_distribution(df)

    if "source1" in dfs:
        for other_key in ["source2", "source3"]:
            if other_key in dfs:
                exact_overlap_check(dfs["source1"], dfs[other_key], other_key.upper())

    if "ground_truth" in dfs:
        ground_truth_profile(dfs["ground_truth"])

    print("\n\n===== DONE. Copy this entire output and paste it back. =====")


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--root ROOT] [--split {train,test}]
                             [--sample SAMPLE]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\chinm\AppData\Roaming\jupyter\runtime\kernel-v3946a5915cd816d89eff2e48cbd259228a79b2411.json


SystemExit: 2

c:\ProgramData\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# Jupyter-safe runner: argparse should see only this program's arguments.
import sys

_saved_argv = sys.argv
try:
    sys.argv = ["inspect_data.py"]
    main()
finally:
    sys.argv = _saved_argv